In [1]:
import os
from utils.data_utils import load_metadata_and_map
from cuml.cluster import KMeans
import numpy as np
import torch
import pandas as pd
from utils import utils
from utils.utils import load_tensor

In [2]:
METADATA_FILE = "embeddings/id_metadata.json"
TENSOR_DIRECTORY = "embeddings/uuid_embeddings/embeddings/"

In [3]:

tensors = load_tensor(TENSOR_DIRECTORY, num_workers=16)


Loading cache from embeddings/cache


In [4]:
metadata_df, artist_to_track_idx = load_metadata_and_map(METADATA_FILE, tensors)

Loading metadata from embeddings/id_metadata.json...
Building Artist to Track Index map...


In [5]:
all_embeddings = utils.pool_loaded_tensor_dict(tensors=tensors, mode='mean+max')

embeddings_torch = torch.stack(list(all_embeddings.values())).cuda()

Detected allchunks tensors, pooling to 1D embedding (mean+max)


100%|██████████| 138748/138748 [00:04<00:00, 30801.14it/s]


In [6]:
N_CLUSTERS = 50

In [7]:
# 4. KMeans Clustering
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init="auto", random_state=1)
kmeans.fit(embeddings_torch)

cluster_labels = kmeans.labels_
codebook = kmeans.cluster_centers_

In [8]:
import torch
import numpy as np
import cupy as cp
import cupyx
from cuml.cluster import KMeans
from collections import defaultdict
from typing import Dict


# 5. Compute VLAD Vectors
# Prepare GPU arrays for calculation
all_embeddings_gpu = cp.asarray(embeddings_torch)
all_codebook_gpu = cp.asarray(codebook)
embedding_dim = embeddings_torch.shape[1]

# Initialize container for accumulated residuals
# shape: (N_CLUSTERS, dim)
artist_vlad_matrices: Dict[str, cp.ndarray] = defaultdict(
  lambda: cp.zeros((N_CLUSTERS, embedding_dim), dtype=cp.float32)
)

# Scatter Add Loop
for artist_id, indices in artist_to_track_idx.items():
  indices_gpu = cp.asarray(indices)
  
  # Get tracks and labels for this artist
  track_embeddings = all_embeddings_gpu[indices_gpu]
  
  # Map global labels to local selection
  # Note: cluster_labels might be on GPU, so we slice safely
  local_labels = cp.asarray(cluster_labels[indices])
  
  assigned_centroids = all_codebook_gpu[local_labels]
  residuals = track_embeddings - assigned_centroids

  # Accumulate residuals into the VLAD matrix for this artist
  cupyx.scatter_add(artist_vlad_matrices[artist_id], local_labels, residuals)


In [9]:
# 6. Normalize (Power + L2)
vlad_vectors: Dict[str, cp.ndarray] = {}

for artist_id, mat in artist_vlad_matrices.items():
  # Power normalization: sign(x) * sqrt(|x|)
  mat = cp.sign(mat) * cp.sqrt(cp.abs(mat))
  
  # Flatten (K * D)
  vec = mat.flatten()
  
  # L2 Normalization
  norm = cp.linalg.norm(vec)
  if norm > 1e-7:
    vec = vec / norm
      
  vlad_vectors[artist_id] = vec

In [10]:
# 7. Search
from utils.data_utils import knn_search, get_artist_name

QUERY_ARTIST_ID = '85de2a90-ea44-473f-af2f-2c22dcc99064'

print(f"Searching 10 nearest neighbors for {QUERY_ARTIST_ID} (VLAD)...")
if QUERY_ARTIST_ID not in vlad_vectors:
  print(f"Error: Artist ID {QUERY_ARTIST_ID} not found.")


# Note: We pass backend=cp to handle the GPU arrays
result_ids = knn_search(
  vlad_vectors, 
  vlad_vectors[QUERY_ARTIST_ID], 
  k=10, 
  backend=cp
)

for res_id in result_ids:
  print(f"- {get_artist_name(metadata_df, res_id)}")


Searching 10 nearest neighbors for 85de2a90-ea44-473f-af2f-2c22dcc99064 (VLAD)...
- TAMUSIC
- 弦奏楽団
- Re：Volte
- ひよこ印の音楽屋さん
- そなたび
- 氷結幻想
- 光と闇の協奏曲
- 針の音楽
- 幻奏盛宴·幻想交响音乐会
- Unknown-Dimension


In [ ]:
val_data = pd.read_json("data/validation/top_1000_artists_by_variance.json")
val_data.head()

,Id,Name
0,00a577e0-ccd9-4463-8fc6-f77ae6e99d53,Pure Highball
1,af62098a-1ffd-499b-bc2d-4d1b2bcc33ad,アールグレイ
2,fafe73a1-168a-4855-9e51-e070182a1bd2,Re：Volte
3,3e72f0ef-c795-4573-9265-13e3c1298e1c,まさかど☆くらいしすっ！
4,a9d6f2a9-55cd-4286-a041-60993b1e3eea,2nd Flush
...,...,...
609,faa689b4-b345-4e0d-be19-066eebf55a56,岸田教団&THE明星ロケッツ
610,edc476dd-b993-47ce-bec6-2aae1dce30d3,FalKKonE
611,b08bb9b5-4e2e-45e2-9e39-eb65bea3ee49,鉄腕トカゲ探知機
612,c2e3595f-672e-4cb8-a7ff-e984735370d8,A-GEAR


In [12]:
results = []

import tqdm

print("Generating results for validation set...")
for _, row in tqdm.tqdm(val_data.iterrows()):
  artist_id = row["Id"]
  artist_name = row["Name"]
  if artist_id not in vlad_vectors:
    print(f"Error: Artist ID {artist_id} not found.")
    continue
  result_ids = knn_search(
    vlad_vectors, 
    vlad_vectors[artist_id], 
    k=10, 
    backend=cp
  )

  results.append({
    "ArtistID": artist_id,
    "Name": artist_name,
    "Results": result_ids
  })

results = pd.DataFrame(results)

results.head()

Generating results for validation set...


614it [00:23, 26.39it/s]


,ArtistID,Name,Results
0,00a577e0-ccd9-4463-8fc6-f77ae6e99d53,Pure Highball,"[00a577e0-ccd9-4463-8fc6-f77ae6e99d53, a185bd0..."
1,af62098a-1ffd-499b-bc2d-4d1b2bcc33ad,アールグレイ,"[af62098a-1ffd-499b-bc2d-4d1b2bcc33ad, 5c48538..."
2,fafe73a1-168a-4855-9e51-e070182a1bd2,Re：Volte,"[fafe73a1-168a-4855-9e51-e070182a1bd2, 85de2a9..."
3,3e72f0ef-c795-4573-9265-13e3c1298e1c,まさかど☆くらいしすっ！,"[3e72f0ef-c795-4573-9265-13e3c1298e1c, f835431..."
4,a9d6f2a9-55cd-4286-a041-60993b1e3eea,2nd Flush,"[a9d6f2a9-55cd-4286-a041-60993b1e3eea, 5c79579..."


In [13]:
# save results
results.to_json("data/validation/vlad_results.jsonl", orient="records", lines=True)
